In [ ]:
from utils.postgres_util import get_engine_from_env, read_sql_dataframe
from utils.final_aligned_observation_writer import build_final_aligned_observations_stage


In [ ]:
SCHEMA = str("capstone")

DATASET_ID = str("pump_synthetic_v1")
RUN_ID = str("premelt_run_001")

NUMBER_OF_SENSORS = int(52)


IF_EXISTS_FLAG = str("replace")
COMPLETE_ONLY_FLAG = True
PREFER_REBUILT_SENSOR_VALUES_FLAG = True

OBSERVATION_WINDOW_SIZE = int(2500)



PREMELT_SOURCE_TABLE_NAME = str("synthetic_observations_premelt_stage")
REBUILT_DESTINATION_TABLE_NAME = str("synthetic_sensor_observations_rebuilt_stage")
TARGET_TABLE_NAME = str("synthetic_sensor_observations_final_aligned_stage")


In [ ]:
engine = get_engine_from_env()

In [ ]:
final_aligned_result = build_final_aligned_observations_stage(
    engine=engine,
    schema=SCHEMA,
    premelt_table=PREMELT_SOURCE_TABLE_NAME,
    rebuilt_table=REBUILT_DESTINATION_TABLE_NAME,
    target_table=TARGET_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    n_sensors=NUMBER_OF_SENSORS,
    complete_only=COMPLETE_ONLY_FLAG,
    prefer_rebuilt_sensor_values=PREFER_REBUILT_SENSOR_VALUES_FLAG,
    if_exists=IF_EXISTS_FLAG,
    observation_window_size=OBSERVATION_WINDOW_SIZE,
)

In [ ]:

final_aligned_result

----

In [ ]:
validation_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        COUNT(*) AS row_count,
        MIN(batch_id) AS min_batch_id,
        MAX(batch_id) AS max_batch_id,
        MIN(row_in_batch) AS min_row_in_batch,
        MAX(observation_index) AS max_observation_index,
        COUNT(*) FILTER (WHERE rebuild_is_complete = TRUE) AS complete_rows,
        MIN(observation_timestamp) AS min_observation_timestamp,
        MAX(observation_timestamp) AS max_observation_timestamp
    FROM capstone.synthetic_sensor_observations_final_aligned_stage
    """
)


In [ ]:

display(validation_dataframe)

----

In [ ]:
sample_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        batch_id,
        row_in_batch,
        global_cycle_id,
        stream_state,
        phase,
        created_at,
        meta_episode_id,
        meta_primary_fault_type,
        meta_magnitude,
        sensor_00,
        sensor_01,
        sensor_02,
        sensor_03,
        sensor_04,
        dataset_id,
        run_id,
        asset_id,
        generated_row_id,
        observation_index,
        observation_timestamp,
        rebuild_is_complete
    FROM capstone.synthetic_sensor_observations_final_aligned_stage
    ORDER BY batch_id, row_in_batch
    LIMIT 10
    """
)


In [ ]:

display(sample_dataframe)